> 📘 **전체 방법·이슈 정리:** [`DRAWING_EXTRACTION_GUIDE.md`](./DRAWING_EXTRACTION_GUIDE.md)
> 표제란 5필드 추출(섹션 1~5), 단일 `cell` YOLO 매핑(섹션 6), 치수·공차·GD&T·반경 OBB 파이프라인(섹션 7)과
> 학습 셋업(`drawing_obb.yaml`, `train_obb.py`)·주의사항을 한 문서로 정리해 두었습니다.

# Drawing Title-Block Extraction with Qwen2.5-VL-7B — ⚠️ 베이스라인(환각 비교용)

Extracts five fields from an engineering-drawing title block into strict JSON:
**Title, Drawing No., LIC. Material, Material, Rev**.

> ⚠️ **이 노트북은 환각(hallucination) 비교용 베이스라인입니다.**
> 표제란 크롭/확대(ROI), Rev 전용 크롭, LIC. Material 빈칸 판별 등
> 환각을 줄이기 위한 전처리 장치를 **모두 제거**하고, **원본 이미지 전체**를
> 그대로 모델에 넣어 5개 필드를 한 번에 읽습니다. 따라서 좁은 Rev 열 누락,
> 빈 칸에 옆 값 복사 등 환각이 그대로 나타날 수 있습니다.
> (전처리 적용 버전은 `extract_title_block.ipynb` 참고)

- Reads each cell **verbatim** (text as printed on the drawing).
- Applies a case-only fix for known material codes (e.g. `SCR18N8` -> `SCr18N8`).
- The model is loaded **once** (section 3); re-run `extract()` for any number of images.

Run with the `kardi_env` kernel.

## 1. Imports & configuration


In [ ]:
import json
import os
import re

import torch
from PIL import Image
# Qwen2.5-VL 모델 클래스와 전처리기(프로세서)를 가져온다.
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
# 이미지/비디오 입력을 모델이 받을 수 있는 형태로 변환해 주는 Qwen 전용 유틸.
from qwen_vl_utils import process_vision_info

# 프로젝트 경로 / 모델 ID / 기본 입력 이미지 경로 설정.
PROJECT_DIR = "/home/jhkim/projects/Title_Qwen_VL_25_7B"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # 사용할 비전-언어 모델

# 입력 도면 이미지는 input_doc/ 에서 읽고, 추출 결과 JSON 은 output/ 에 저장한다.
INPUT_DIR = os.path.join(PROJECT_DIR, "input_doc")   # 입력 도면 이미지 폴더
OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")     # 추출 결과 JSON 폴더
os.makedirs(OUTPUT_DIR, exist_ok=True)               # 없으면 생성

DEFAULT_IMAGE = os.path.join(INPUT_DIR, "test_title_02.png")  # 기본 테스트 이미지(표제란)

# ※ 이 노트북은 '환각(hallucination) 비교용' 베이스라인 버전입니다.
#   표제란 크롭/확대(ROI), Rev 전용 크롭, LIC. Material 빈칸 판별 등
#   환각을 줄이기 위한 전처리 장치를 모두 제거하고, '원본 이미지 전체'를
#   그대로 모델에 넣어 5개 필드를 한 번에 읽게 합니다.


## 2. Prompt & helper functions


In [ ]:
# ── 모델에 보낼 지시문(프롬프트) ─────────────────────────────────────────────
# 도면 표제란(title block)에서 5개 필드를 추출해 "엄격한 JSON" 형식으로만
# 출력하도록 모델에게 지시한다. 핵심 규칙:
#   - 지정한 5개 키만 사용 (Title, Drawing No., LIC. Material, Material, Rev)
#   - 도면에 인쇄된 글자를 그대로(verbatim) 옮길 것 (★공백·괄호까지 보존)
#   - "Rev"는 맨 오른쪽 'Rev' 열에 세로로 쌓인 '모든' 값을 위→아래 배열로 (예: ["1","0"])
#     ※ 맨 아래(최신) 값 하나만 반환하지 말 것
#   - 별도의 "Security" 칸 값(예: "B")과 혼동 금지
#   - 찾지 못한 필드는 null / JSON 객체만 출력 (마크다운/설명 금지)
PROMPT = (
    "This image is the title block of a mechanical engineering drawing.\n"
    "Extract exactly the following five fields and return them as strict JSON:\n"
    '  "Title"         - the part/drawing title (the large text in the "Title" cell)\n'
    '  "Drawing No."   - the drawing number (the "Drawing No." cell)\n'
    '  "LIC. Material" - the value in the "LIC. Material / Blank" cell\n'
    '  "Material"      - the value in the "Material / Blank" cell '
    '(the one WITHOUT the "LIC." prefix)\n'
    '  "Rev"           - The far-right column titled "Rev" can contain MULTIPLE '
    'revision values stacked vertically (one per revision row). Return EVERY value '
    'present in that column as a JSON array of strings, ordered TOP to BOTTOM '
    '(e.g. if the column shows 1 above 0, return ["1","0"]). '
    'Read each revision row and include its Rev value -- do NOT return only the '
    'latest/bottom value, and do NOT collapse them into one. '
    'Each entry is a single digit/letter. If there is genuinely only one value, '
    'return a 1-element array (e.g. ["0"]). '
    'Do NOT use the separate "Security" cell value (e.g. "B") for this field.\n'
    "Rules:\n"
    "- Use these exact JSON keys: \"Title\", \"Drawing No.\", "
    "\"LIC. Material\", \"Material\", \"Rev\".\n"
    "- Copy the text VERBATIM as printed, preserving every space and punctuation "
    "mark EXACTLY. Keep the space before '(' and the spaces inside the text "
    '(e.g. "FLANGE (JIS 1K-1000A)" -- do NOT output "FLANGE(JIS1K-1000A)"). '
    "Never remove, add, or normalize spaces.\n"
    "- If a field cannot be found, set its value to null.\n"
    "- Output ONLY the JSON object. No markdown fences, no explanation."
)

# ── 알려진 재질 코드의 표준(정식) 철자 목록 ──────────────────────────────────
# 모델 OCR 결과가 아래 값과 "대소문자만 다르게" 일치하면 표준 철자로 교정한다.
# 이 보정은 오직 '대소문자'만 바꾸며, 글자 자체(문자 구성)는 절대 바꾸지 않는다.
# 목록에 없는 값은 그대로 통과시킨다. 새 코드는 여기에 추가하면 된다.
CANONICAL_MATERIALS = [
    "SCr18N8",
    "SUS316L",
    "SCrNi",
]


def fix_material_case(value):
    """알려진 재질 코드를 표준 대소문자로 교정한다(대소문자만 변경)."""
    if not isinstance(value, str):
        return value
    # 앞뒤 공백 제거 + 대소문자 무시 비교를 위한 키 생성.
    key = value.strip().casefold()
    for canon in CANONICAL_MATERIALS:
        # 대소문자 무시하고 일치하면, 표준 철자(canon)로 바꿔서 반환.
        if canon.casefold() == key:
            return canon
    return value  # 일치하는 표준 코드가 없으면 원본 그대로


def parse_json(text):
    """모델 출력에서 첫 번째 {...} JSON 객체만 뽑아내 파싱한다."""
    # 혹시 모델이 ```json ... ``` 같은 코드펜스를 붙였다면 제거.
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    # 가장 바깥쪽 중괄호 { ... } 구간만 잘라낸다(앞뒤 잡텍스트 방어).
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start : end + 1]
    return json.loads(cleaned)  # 문자열 → 파이썬 dict

## 3. Load the model (run once)

First run downloads/loads ~16.5 GB across 5 shards (~1-2 min).

In [ ]:
print(f"Loading {MODEL_ID} ...")
# 모델 로드: torch_dtype="auto"는 가중치에 맞는 정밀도 자동 선택,
# device_map="auto"는 사용 가능한 GPU(들)에 자동으로 레이어를 배치한다.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype="auto", device_map="auto"
)
# 프로세서: 텍스트 토크나이저 + 이미지 전처리를 함께 담당한다.
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model ready on:", model.device)

## 4. Extraction function


In [ ]:
def extract(image_path=DEFAULT_IMAGE, save=True):
    """도면(표제란) 이미지 1장에서 표제란 필드를 추출해 dict로 반환한다.

    ※ 환각 비교용 베이스라인: 크롭/확대/Rev·LIC 전용 읽기 없이
       '원본 이미지 전체'를 그대로 모델에 넣어 5개 필드를 한 번에 읽는다.
    """
    if not os.path.isfile(image_path):
        raise FileNotFoundError(image_path)

    # ── 0) 입력 이미지 준비 (크롭/확대 없이 원본 그대로) ─────────────────
    image = Image.open(image_path).convert("RGB")

    # ── 1) 대화 메시지 구성: 원본 이미지 + 텍스트 프롬프트 ───────────────
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},  # 원본 PIL 이미지 그대로 전달
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    # ── 2) 입력 전처리 ──────────────────────────────────────────────────
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # ── 3) 추론(텍스트 생성) ────────────────────────────────────────────
    with torch.no_grad():  # 추론만 하므로 그래디언트 계산 비활성화(메모리 절약)
        generated = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    # ── 4) 후처리: JSON 파싱 + 재질 코드 대소문자 교정 ─────────────────────
    result = parse_json(raw)

    for field in ("LIC. Material", "Material"):
        if field in result:
            fixed = fix_material_case(result[field])
            if fixed != result[field]:
                print(f'{field} case-fixed: "{result[field]}" -> "{fixed}"')
                result[field] = fixed

    # ── 5) 결과 저장(선택) ──────────────────────────────────────────────
    if save:
        stem = os.path.splitext(os.path.basename(image_path))[0]
        # ★ 전처리 버전(extract_title_block.ipynb)의 '{stem}_title_block.json' 과
        #   덮어쓰기 충돌을 막기 위해 베이스라인은 '_baseline' 접미사로 따로 저장.
        output_json = os.path.join(OUTPUT_DIR, f"{stem}_title_block_baseline.json")
        with open(output_json, "w", encoding="utf-8") as fh:
            fh.write(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
        print(f"Saved -> {output_json}")

    return result

## 5. Run


In [ ]:
# 도면 1장씩: '모델 입력 이미지(원본) → 추출 JSON' 을 번갈아 출력한다.
#   (이미지1 → JSON1 → 이미지2 → JSON2 → 이미지3 → JSON3)
#   각 결과 JSON 은 output/ 에도 저장된다.
from IPython.display import display

# 처리할 표제란 이미지 3장 (input_doc/ 안의 파일명).
TITLE_IMAGES = [
    os.path.join(INPUT_DIR, "test_title_01.png"),
    os.path.join(INPUT_DIR, "test_title_02.png"),
    os.path.join(INPUT_DIR, "test_title_03.png"),
]

all_results = {}
for image_path in TITLE_IMAGES:
    name = os.path.basename(image_path)
    print(f"\n===== {name} =====")

    # 1) 모델에 실제로 들어가는 '원본 이미지' 출력 (크롭/확대 없음)
    img = Image.open(image_path).convert("RGB")
    print(f"모델 입력 크기: {img.size[0]}x{img.size[1]} px")
    display(img)

    # 2) 그 이미지에서 추출한 JSON 출력
    result = extract(image_path)
    all_results[name] = result
    print(json.dumps(result, ensure_ascii=False, indent=2))


## 5b. 두 버전 결과 비교 (전처리 vs 베이스라인)

두 노트북을 각각 실행해 결과 JSON이 모두 생긴 뒤 아래 셀을 실행하면 필드별 차이를 표로 보여준다.

- **전처리 버전** (`extract_title_block.ipynb`): `output/{stem}_title_block.json`
- **베이스라인/환각** (이 노트북): `output/{stem}_title_block_baseline.json`

각 필드를 비교해 값이 다르면 `≠` 로 표시한다.

In [ ]:
# ── 전처리 버전 vs 베이스라인(환각) 결과 자동 비교 ───────────────────────────
#   전처리 : output/{stem}_title_block.json          (extract_title_block.ipynb 가 저장)
#   베이스라인: output/{stem}_title_block_baseline.json  (이 노트북이 저장)
#   두 파일이 모두 있어야 비교됨(없으면 '(파일 없음)' 으로 표시).
FIELDS = ["Title", "Drawing No.", "LIC. Material", "Material", "Rev"]


def _load_json(path):
    """JSON 파일을 읽어 dict 반환. 없으면 None."""
    if not os.path.isfile(path):
        return None
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def _fmt(value):
    """비교 표에 넣을 값 문자열화 (None/리스트도 보기 좋게)."""
    if value is None:
        return "null"
    if isinstance(value, list):
        return "[" + ", ".join(map(str, value)) + "]"
    return str(value)


def compare_results(image_paths=None):
    """이미지별로 전처리 vs 베이스라인 JSON을 필드 단위로 비교 출력한다."""
    if image_paths is None:
        image_paths = TITLE_IMAGES

    n_diff_total = 0
    for image_path in image_paths:
        stem = os.path.splitext(os.path.basename(image_path))[0]
        pre = _load_json(os.path.join(OUTPUT_DIR, f"{stem}_title_block.json"))
        base = _load_json(os.path.join(OUTPUT_DIR, f"{stem}_title_block_baseline.json"))

        print(f"\n===== {stem} =====")
        if pre is None or base is None:
            miss = []
            if pre is None:
                miss.append("전처리(_title_block.json)")
            if base is None:
                miss.append("베이스라인(_title_block_baseline.json)")
            print(f"  ⚠️ 비교 불가 — 파일 없음: {', '.join(miss)}")
            continue

        # 표 헤더 (열 너비 고정)
        print(f"  {'Field':<14} | {'전처리':<28} | {'베이스라인(환각)':<28} | 일치")
        print(f"  {'-'*14}-+-{'-'*28}-+-{'-'*28}-+-----")
        for f in FIELDS:
            pv, bv = _fmt(pre.get(f)), _fmt(base.get(f))
            same = pre.get(f) == base.get(f)
            mark = "✓" if same else "≠"
            if not same:
                n_diff_total += 1
            print(f"  {f:<14} | {pv:<28} | {bv:<28} | {mark}")

    print(f"\n총 불일치 필드 수: {n_diff_total}")


compare_results()


## 6. (선택) 2단계 파이프라인: YOLO(단일 `cell` 클래스) → 규칙 매핑 → Qwen 읽기

표제란 **양식이 여러 종이거나 스캔/회전 왜곡**이 있어 칸 위치가 매번 달라질 때 유용한 방식.

- **YOLO**: 생김새로 "글자 칸(cell)"의 **위치 박스**만 검출 (필드명은 맞히지 않음 → 클래스 혼동 없음).
- **규칙 매핑(코드)**: 박스들을 행/열로 정렬하고, **위치 + 인쇄된 항목명(앵커)** 으로 어느 필드인지 결정.
- **Qwen**: 각 칸을 **크롭·확대**해서 "이 작은 칸의 글자만 읽어" → 할루시네이션 최소화.

> 양식이 1~2종으로 고정적이라면 YOLO 없이 **고정 ROI 크롭**이 더 간단하고 정확합니다.
> 아래 코드는 **스캐폴드**이며, `map_fields`의 규칙은 실제 도면 양식에 맞춰 조정해야 합니다 (TODO 표시).

In [ ]:
# ── 6-1. YOLO 검출 (단일 "cell" 클래스) ──────────────────────────────────────
from ultralytics import YOLO  # 설치: pip install ultralytics

# 단일 클래스("cell")로 학습한 YOLO 가중치 경로. 학습 후 생성된 best.pt를 지정.
#   예) yolo detect train data=cell.yaml model=yol11n.pt imgsz=1280 epochs=100
#   cell.yaml 에는 names: ["cell"] 단일 클래스만 정의.
YOLO_WEIGHTS = os.path.join(PROJECT_DIR, "yolo_cell", "weights", "best.pt")

# 가중치가 아직 없으면 None (구조만 확인하고 detect_cells 호출 시 안내 에러).
yolo_model = YOLO(YOLO_WEIGHTS) if os.path.isfile(YOLO_WEIGHTS) else None


def detect_cells(image_path, conf=0.25):
    """YOLO로 표제란의 글자 칸(cell) 박스들을 검출한다.

    반환: [(x1, y1, x2, y2, conf), ...]  좌표는 픽셀 단위.
    YOLO는 '필드명'이 아니라 '칸 위치'만 찾는다 → 필드 매핑은 다음 셀의 규칙이 담당.
    """
    if yolo_model is None:
        raise RuntimeError(
            f"YOLO 가중치가 없습니다: {YOLO_WEIGHTS}\n"
            "단일 'cell' 클래스로 학습한 best.pt 경로를 지정하세요."
        )
    res = yolo_model.predict(image_path, conf=conf, verbose=False)[0]
    boxes = []
    for b in res.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        boxes.append((x1, y1, x2, y2, float(b.conf[0])))
    return boxes

In [ ]:
# ── 6-2. 기하 헬퍼 + 행/열 정렬 + 필드 매핑 규칙 ─────────────────────────────

def _center(box):
    """박스의 중심 좌표 (cx, cy)를 반환."""
    x1, y1, x2, y2, *_ = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def group_into_rows(boxes, y_tol_ratio=0.5):
    """박스들을 위→아래 '행(row)'으로 묶고, 각 행은 왼→오른쪽으로 정렬한다.

    표제란은 격자(표) 구조이므로, y 중심이 비슷하면 같은 행으로 본다.
    y_tol_ratio: 같은 행으로 인정할 y 허용오차(칸 높이 중앙값 대비 비율).
    """
    if not boxes:
        return []
    heights = sorted((b[3] - b[1]) for b in boxes)
    med_h = heights[len(heights) // 2]          # 칸 높이의 중앙값
    y_tol = med_h * y_tol_ratio

    boxes_sorted = sorted(boxes, key=lambda b: _center(b)[1])  # y 기준 정렬
    rows, cur = [], [boxes_sorted[0]]
    for b in boxes_sorted[1:]:
        # 직전 행 박스와 y 중심 차이가 허용오차 이내면 같은 행으로 누적.
        if abs(_center(b)[1] - _center(cur[-1])[1]) <= y_tol:
            cur.append(b)
        else:
            rows.append(cur)
            cur = [b]
    rows.append(cur)
    return [sorted(r, key=lambda b: _center(b)[0]) for r in rows]  # 행 내부 x 정렬


def _norm(text):
    """앵커(항목명) 비교용: 공백 제거 + 소문자화."""
    return re.sub(r"\s+", "", text or "").casefold()


def _right_neighbor(target_box, cells):
    """target_box 와 같은 행에서 '바로 오른쪽' 칸의 (box, text)를 찾는다."""
    tcx, tcy = _center(target_box)
    cands = []
    for box, text in cells:
        if box is target_box:
            continue
        cx, cy = _center(box)
        h = box[3] - box[1]
        if abs(cy - tcy) <= h * 0.6 and cx > tcx:   # 같은 행 & 오른쪽
            cands.append((cx, box, text))
    cands.sort(key=lambda c: c[0])
    return (cands[0][1], cands[0][2]) if cands else None


def _below_neighbor(target_box, cells):
    """target_box '바로 아래'(x가 겹치는) 칸의 (box, text)를 찾는다."""
    tcx, tcy = _center(target_box)
    cands = []
    for box, text in cells:
        if box is target_box:
            continue
        cx, cy = _center(box)
        w = box[2] - box[0]
        if abs(cx - tcx) <= w * 0.6 and cy > tcy:   # x 겹침 & 아래
            cands.append((cy, box, text))
    cands.sort(key=lambda c: c[0])
    return (cands[0][1], cands[0][2]) if cands else None


def map_fields(cells):
    """읽은 칸 목록 [(box, text), ...] 을 5개 필드로 매핑한다.

    전략 = 위치 규칙 + 항목명 앵커:
      (1) 항목명이 인쇄된 '앵커 칸'을 키워드로 찾고,
      (2) 그 값은 같은 행의 오른쪽 칸 또는 바로 아래 칸에서 가져온다.
    아래 규칙은 예시 스캐폴드 — 실제 도면 양식에 맞춰 조정할 것 (TODO).
    """
    result = {k: None for k in
              ("Title", "Drawing No.", "LIC. Material", "Material", "Rev")}
    rows = group_into_rows([b for b, _ in cells])

    for box, text in cells:
        n = _norm(text)

        # TODO: 양식별 항목명 표기에 맞춰 키워드 보강.
        # 'LIC.' 앵커가 보이면 그 값은 같은 행 오른쪽 칸.
        if "lic." in n and "material" in n:
            nb = _right_neighbor(box, cells)
            if nb:
                result["LIC. Material"] = nb[1]
        # 'LIC.' 없는 'Material' 앵커 → 일반 재질.
        elif "material" in n:
            nb = _right_neighbor(box, cells)
            if nb:
                result["Material"] = nb[1]
        elif "drawing" in n and "no" in n:
            nb = _right_neighbor(box, cells)
            if nb:
                result["Drawing No."] = nb[1]
        elif n == "title" or n.startswith("title"):
            nb = _right_neighbor(box, cells) or _below_neighbor(box, cells)
            if nb:
                result["Title"] = nb[1]

    # Rev: 항목명이 없을 때가 많음 → 위치 규칙으로 보완.
    # 맨 오른쪽 열의 '가장 아래' 좁은 칸을 Rev 후보로 사용. (TODO: 양식 확인)
    if result["Rev"] is None and rows:
        last_row = rows[-1]
        if last_row:
            rightmost = max(last_row, key=lambda b: _center(b)[0])
            for box, text in cells:
                if box is rightmost:
                    result["Rev"] = text
                    break

    return result

In [ ]:
# ── 6-3. 칸 크롭·확대 → Qwen으로 한 칸 읽기 → 전체 오케스트레이션 ──────────────
from PIL import Image

UPSCALE = 3  # 작은 글씨(Rev, 재질 코드 등) OCR 정확도를 위해 크롭을 확대.

# 단일 칸 읽기 전용 프롬프트: '추론'이 아니라 '그대로 받아쓰기'만 시킴 → 환각 최소화.
CELL_READ_PROMPT = (
    "Transcribe the text printed inside this image verbatim. "
    "Output ONLY the raw text, no quotes, no explanation. "
    "If it is empty, output nothing."
)


def crop_cell(pil_img, box, pad=2):
    """박스 영역을 약간의 여백과 함께 크롭하고 UPSCALE 배 확대한다."""
    x1, y1, x2, y2, *_ = box
    x1, y1 = max(0, int(x1) - pad), max(0, int(y1) - pad)
    x2, y2 = int(x2) + pad, int(y2) + pad
    crop = pil_img.crop((x1, y1, x2, y2))
    if UPSCALE != 1:
        crop = crop.resize((crop.width * UPSCALE, crop.height * UPSCALE))
    return crop


def read_cell(pil_crop):
    """크롭된 한 칸 이미지를 Qwen에게 넘겨 안의 글자만 그대로 읽는다.

    (셀 3에서 로드한 model/processor를 그대로 재사용)
    """
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil_crop},   # PIL 이미지 직접 전달 가능
            {"type": "text", "text": CELL_READ_PROMPT},
        ],
    }]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()


def extract_with_yolo(image_path=DEFAULT_IMAGE, save=True):
    """YOLO 검출 → 칸별 Qwen 읽기 → 규칙 매핑으로 5개 필드를 추출한다."""
    pil_img = Image.open(image_path).convert("RGB")

    boxes = detect_cells(image_path)                       # 1) 칸 위치 검출
    cells = [(b, read_cell(crop_cell(pil_img, b))) for b in boxes]  # 2) 칸별 읽기
    result = map_fields(cells)                             # 3) 위치/앵커 규칙 매핑

    # 4) 재질 코드 대소문자 보정 (기존 헬퍼 재사용).
    for field in ("LIC. Material", "Material"):
        if result.get(field):
            result[field] = fix_material_case(result[field])

    if save:
        stem = os.path.splitext(os.path.basename(image_path))[0]
        output_json = os.path.join(OUTPUT_DIR, f"{stem}_title_block_yolo.json")
        with open(output_json, "w", encoding="utf-8") as fh:
            fh.write(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
        print(f"Saved -> {output_json}")

    return result

In [ ]:
# ── 6-4. 실행 (YOLO 가중치 준비 후) ──────────────────────────────────────────
# best.pt 를 준비한 뒤 실행. 디버깅 시 검출 박스/읽은 텍스트를 함께 확인하면 좋다.
result_yolo = extract_with_yolo(DEFAULT_IMAGE)
print(json.dumps(result_yolo, ensure_ascii=False, indent=2))

## 7. (선택) 도면 주석 추출: YOLO **OBB** → 회전보정 크롭 → Qwen 구조화 파싱

치수 · 공차 · GD&T · 반경을 뽑는 파이프라인. (참고 이미지 `doc_sample_paper.jpg`의 분류 체계와 동일)
표제란과 **방법론은 같지만**(검출→크롭→VLM 읽기) 세 가지가 다름:

1. **OBB(회전 박스)** — 치수/반경/GD&T는 기울어져 인쇄되는 경우가 많아(예: `R42`) 수평 박스(AABB)로 자르면 글자가 섞임 → 회전 박스로 검출 후 **수평으로 펴서(deskew)** 크롭.
2. **3종 클래스** (도면 범례와 동일) — `measure`(치수값+공차) / `gdt`(기하공차 프레임) / `radii`(반경 R 치수).
3. **클래스별 구조화 출력** — Qwen이 크롭마다 strict JSON 반환:
   - measure → `{nominal, upper, lower, raw}`  (예: `Ø36 H8`, `18 ±0,1`, `20 +0,3/0`)
   - gdt → `{characteristic, symbol, tolerance, modifier, datums[], raw}` (프레임 구성요소 분해)
   - radii → `{nominal, upper, lower, raw}`  (예: `R3`, `R42`)

> 스캐폴드입니다. OBB 가중치(`best.pt`), 클래스 순서(`CLASS_NAMES`), `deskew_crop`의 회전 부호는 실제 데이터로 검증/조정하세요 (TODO 표시).

In [ ]:
# ── 7-1. YOLO OBB 검출 (3종 클래스: measure / gdt / radii) ────────────────────
import cv2
import numpy as np
from ultralytics import YOLO  # OBB 지원: YOLOv8/YOLO11 obb 모델

# 3종 클래스로 학습한 OBB 가중치. 학습 명령 예:
#   yolo obb train data=drawing_obb.yaml model=yolo11n-obb.pt imgsz=1280 epochs=100
# 도면 범례(Measure / GD&T / Radii)와 동일한 분류.
# drawing_obb.yaml 의 names 순서와 아래 CLASS_NAMES 인덱스를 반드시 일치시킬 것!
OBB_WEIGHTS = os.path.join(PROJECT_DIR, "yolo_obb_drawing", "weights", "best.pt")
CLASS_NAMES = {0: "measure", 1: "gdt", 2: "radii"}  # TODO: data.yaml names 순서와 동일하게

obb_model = YOLO(OBB_WEIGHTS) if os.path.isfile(OBB_WEIGHTS) else None


def detect_obb(image_path, conf=0.25):
    """OBB(회전 박스) 검출.

    반환: [{"cls", "name", "conf", "poly"(4코너 [[x,y]x4]), "xywhr"(cx,cy,w,h,라디안)}]
    """
    if obb_model is None:
        raise RuntimeError(
            f"OBB 가중치가 없습니다: {OBB_WEIGHTS}\n"
            "3종 클래스(measure/gdt/radii) OBB로 학습한 best.pt 경로를 지정하세요."
        )
    res = obb_model.predict(image_path, conf=conf, verbose=False)[0]
    dets = []
    if res.obb is None:          # OBB 모델이 아니면 None
        return dets
    for o in res.obb:
        cls = int(o.cls[0])
        dets.append({
            "cls": cls,
            "name": CLASS_NAMES.get(cls, str(cls)),
            "conf": float(o.conf[0]),
            "poly": o.xyxyxyxy[0].tolist(),   # 4개 코너 좌표(픽셀)
            "xywhr": o.xywhr[0].tolist(),      # 중심x, 중심y, 너비, 높이, 회전각(rad)
        })
    return dets

In [ ]:
# ── 7-2. 회전 보정(deskew) 크롭 ──────────────────────────────────────────────
def deskew_crop(image_bgr, xywhr, pad=4, upscale=2):
    """OBB의 회전각을 보정해 글자를 '수평으로 펴서' 크롭한 PIL 이미지를 반환.

    원리: 박스 중심 (cx,cy)을 기준으로 이미지 전체를 회전각만큼 되돌린 뒤,
    중심 주변 w×h 영역을 똑바로 잘라낸다.
    """
    cx, cy, w, h, ang = xywhr
    deg = float(np.degrees(ang))

    # 중심을 축으로 이미지를 deg 만큼 회전(박스를 수평으로). 부호가 반대로 펴지면
    # deg → -deg 로 바꿀 것 (학습 좌표계에 따라 다름). TODO: 실제 결과 보고 확정.
    H, W = image_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((cx, cy), deg, 1.0)
    rotated = cv2.warpAffine(image_bgr, M, (W, H), flags=cv2.INTER_CUBIC,
                             borderValue=(255, 255, 255))

    # 회전 후 박스는 (cx,cy) 중심의 수평 사각형이 됨 → 여백 포함해 잘라냄.
    bw, bh = int(round(w)) + 2 * pad, int(round(h)) + 2 * pad
    x0 = max(0, int(round(cx - bw / 2)))
    y0 = max(0, int(round(cy - bh / 2)))
    crop = rotated[y0:y0 + bh, x0:x0 + bw]
    if crop.size == 0:
        return None

    # 세로로 더 긴 박스(세로쓰기 치수)는 가로로 눕히면 OCR에 유리할 수 있음.
    if crop.shape[0] > crop.shape[1] * 1.5:   # 높이 > 너비*1.5 → 세로형으로 판단
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)  # TODO: 방향 검증

    # 작은 글씨 가독성 향상을 위해 확대.
    if upscale != 1:
        crop = cv2.resize(crop, (crop.shape[1] * upscale, crop.shape[0] * upscale),
                          interpolation=cv2.INTER_CUBIC)

    # cv2(BGR) → PIL(RGB) 변환해 Qwen 입력으로 사용.
    return Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

In [ ]:
# ── 7-3. 클래스별 프롬프트 + Qwen 구조화 읽기 ────────────────────────────────

# measure(치수) / radii(반경) 공통: 공칭치수 / 위·아래 공차 / 원문 으로 분해.
# 참고: 이 도면은 소수점에 쉼표(예: "±0,1", "+0,3")를 쓰므로 raw 는 그대로 보존.
MEASURE_PROMPT = (
    "This is a cropped dimension callout from a mechanical engineering drawing.\n"
    "Return strict JSON with keys: \"nominal\", \"upper\", \"lower\", \"raw\".\n"
    '- "nominal": nominal size incl. any symbol (e.g. "Ø36", "78", "R3", "M8x1.25"). '
    'Keep a fit class with it (e.g. "Ø36 H8").\n'
    '- "upper"/"lower": tolerance limits if shown (e.g. "+0,3", "-0,05", "0"); else null. '
    'For symmetric "±0,1" use upper="+0,1", lower="-0,1". '
    'The decimal separator may be a comma — copy it as printed.\n'
    '- "raw": the full text exactly as printed.\n'
    "Output ONLY the JSON object."
)

# GD&T: 피처컨트롤프레임을 구성요소로 분해.
GDT_PROMPT = (
    "This is a cropped GD&T feature control frame from a mechanical drawing.\n"
    "Return strict JSON with keys: "
    '"characteristic", "symbol", "tolerance", "modifier", "datums", "raw".\n'
    '- "characteristic": control name (e.g. "position", "flatness", '
    '"perpendicularity", "concentricity").\n'
    '- "symbol": the GD&T symbol char if recognizable (e.g. "⌖","⏥","⟂","○"), else null.\n'
    '- "tolerance": tolerance value incl. diameter sign if any (e.g. "Ø0.4", "0.08").\n'
    '- "modifier": material condition on the tolerance if present '
    '("M"=MMC, "L"=LMC), else null.\n'
    '- "datums": ordered list of datum refs incl. their modifier if any '
    '(e.g. ["A","B(M)","C"]); [] if none.\n'
    '- "raw": full text as printed.\n'
    "Output ONLY the JSON object."
)


def _run_vlm(pil_img, prompt, max_new_tokens=128):
    """크롭 PIL 이미지 + 프롬프트로 Qwen 텍스트 생성 (model/processor 재사용)."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil_img},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()


def _read_json(pil_img, prompt):
    """JSON 출력 프롬프트를 실행하고 파싱. 실패 시 {raw, _parse_error} 로 폴백."""
    raw = _run_vlm(pil_img, prompt)
    try:
        return parse_json(raw)                       # 셀 2의 parse_json 재사용
    except Exception as exc:
        return {"raw": raw, "_parse_error": str(exc)}


def read_measure(pil_img):
    """measure / radii 공통 치수 읽기 (반경도 nominal에 'R..' 형태로 들어감)."""
    return _read_json(pil_img, MEASURE_PROMPT)


def read_gdt(pil_img):
    return _read_json(pil_img, GDT_PROMPT)

In [ ]:
# ── 7-4. 전체 오케스트레이션: 검출 → 클래스별 회전보정 크롭 → 구조화 읽기 ──────
def extract_drawing_annotations(image_path=DEFAULT_IMAGE, save=True, conf=0.25):
    """도면에서 치수(measure)·반경(radii)·GD&T를 추출해 클래스별로 정리한 dict 반환."""
    if not os.path.isfile(image_path):
        raise FileNotFoundError(image_path)

    image_bgr = cv2.imread(image_path)        # OpenCV는 BGR 순서로 로드
    dets = detect_obb(image_path, conf=conf)  # 1) OBB 검출

    out = {"measures": [], "radii": [], "gdt": []}
    for d in dets:
        pil_crop = deskew_crop(image_bgr, d["xywhr"])  # 2) 회전보정 크롭
        if pil_crop is None:
            continue

        meta = {"bbox": d["poly"], "conf": round(d["conf"], 3)}  # 위치/신뢰도 함께 보관
        # 3) 클래스별로 알맞은 프롬프트로 읽기 (measure/radii는 동일 치수 파서 사용).
        if d["name"] == "measure":
            out["measures"].append({**read_measure(pil_crop), **meta})
        elif d["name"] == "radii":
            out["radii"].append({**read_measure(pil_crop), **meta})
        elif d["name"] == "gdt":
            out["gdt"].append({**read_gdt(pil_crop), **meta})

    out["counts"] = {k: len(v) for k, v in out.items() if isinstance(v, list)}

    if save:
        stem = os.path.splitext(os.path.basename(image_path))[0]
        output_json = os.path.join(OUTPUT_DIR, f"{stem}_annotations.json")
        with open(output_json, "w", encoding="utf-8") as fh:
            fh.write(json.dumps(out, ensure_ascii=False, indent=2) + "\n")
        print(f"Saved -> {output_json}")

    return out

In [ ]:
# ── 7-5. 실행 (OBB 가중치 준비 후) ───────────────────────────────────────────
# OBB_WEIGHTS(best.pt)를 준비한 뒤 실행.
drawing_path = os.path.join(INPUT_DIR, "doc_sample_paper_01.jpg")
annotations = extract_drawing_annotations(drawing_path)
print("검출 개수:", annotations["counts"])
print(json.dumps(annotations, ensure_ascii=False, indent=2))